# Hillsboro Growth & Development GIS Analysis

## HIL-005 — Buildings

Initial exploration of the City of Hillsboro Buildings GIS REST service.

**Source:** City of Hillsboro GIS  
**Layer:** Planning_BaseData / MapServer / 91  
**Date accessed:** August 25, 2026

### Purpose

Explore the structure, metadata, data quality, and analytical potential of the Hillsboro building footprint dataset before acquiring the complete dataset.

In [ ]:
import requests

url = "https://gis.hillsboro-oregon.gov/public/rest/services/public/Planning_BaseData/MapServer/91"

response = requests.get(
    url,
    params={"f": "json"}
)

data = response.json()

print(data["name"])
print(data["geometryType"])

Buildings (<1,000)
esriGeometryPolygon


In [ ]:
for field in data["fields"]:
    print(field["name"], "—", field["type"])

OBJECTID — esriFieldTypeOID
BLDG_ID — esriFieldTypeInteger
STATUS — esriFieldTypeString
NUM_STORIES — esriFieldTypeSmallInteger
HEIGHT — esriFieldTypeDouble
YEAR_BUILT — esriFieldTypeSmallInteger
SOURCE — esriFieldTypeString
PLANREFID — esriFieldTypeString
CONST_TYPE — esriFieldTypeString
ROOF_TYPE — esriFieldTypeString
ROOF_COVER — esriFieldTypeString
BASEMENT — esriFieldTypeString
SPRINKLED — esriFieldTypeString
NFIRSCD — esriFieldTypeString
OCCUPANCY_TYPE — esriFieldTypeString
Tracking_CreateID — esriFieldTypeString
UTC_CreateDate — esriFieldTypeDate
Tracking_EditID — esriFieldTypeString
UTC_EditDate — esriFieldTypeDate
GlobalID — esriFieldTypeGlobalID
YEAR_DEMOLISHED — esriFieldTypeInteger
DEMO_PERMIT — esriFieldTypeString
Shape — esriFieldTypeGeometry
Shape.STArea() — esriFieldTypeDouble
Shape.STLength() — esriFieldTypeDouble
OMS_FACILITY_ID — esriFieldTypeString
PERMIT_ID — esriFieldTypeString


In [ ]:
count_url = url + "/query"

params = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

count_response = requests.get(count_url, params=params)
count_data = count_response.json()

print("Number of records:", count_data["count"])

Number of records: 43686


In [ ]:
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 5,
    "f": "json"
}

response = requests.get(count_url, params=params)
sample = response.json()

for feature in sample["features"]:
    print(feature["attributes"])

{'OBJECTID': 49347, 'BLDG_ID': 58133, 'STATUS': '0', 'NUM_STORIES': None, 'HEIGHT': 9.68120956, 'YEAR_BUILT': 0, 'SOURCE': 'iTEN2015', 'PLANREFID': None, 'CONST_TYPE': None, 'ROOF_TYPE': None, 'ROOF_COVER': None, 'BASEMENT': None, 'SPRINKLED': None, 'NFIRSCD': None, 'OCCUPANCY_TYPE': None, 'Tracking_CreateID': 'MCLBR', 'UTC_CreateDate': 1549323640000, 'Tracking_EditID': 'CLAAM', 'UTC_EditDate': 1786479151000, 'GlobalID': '{17D1FAA3-24A5-4D62-8088-B17A3C014755}', 'YEAR_DEMOLISHED': None, 'DEMO_PERMIT': None, 'Shape.STArea()': 3588.8831729295403, 'Shape.STLength()': 280.04932376928946, 'OMS_FACILITY_ID': None, 'PERMIT_ID': None}
{'OBJECTID': 49424, 'BLDG_ID': 58210, 'STATUS': '0', 'NUM_STORIES': None, 'HEIGHT': 11.64050006, 'YEAR_BUILT': 1959, 'SOURCE': 'iTEN2015', 'PLANREFID': None, 'CONST_TYPE': None, 'ROOF_TYPE': None, 'ROOF_COVER': None, 'BASEMENT': None, 'SPRINKLED': None, 'NFIRSCD': None, 'OCCUPANCY_TYPE': None, 'Tracking_CreateID': 'MCLBR', 'UTC_CreateDate': 1549323640000, 'Tracki

In [ ]:
import pandas as pd

records = [feature["attributes"] for feature in sample["features"]]

df = pd.DataFrame(records)

df

,OBJECTID,BLDG_ID,STATUS,NUM_STORIES,HEIGHT,YEAR_BUILT,SOURCE,PLANREFID,CONST_TYPE,ROOF_TYPE,...,UTC_CreateDate,Tracking_EditID,UTC_EditDate,GlobalID,YEAR_DEMOLISHED,DEMO_PERMIT,Shape.STArea(),Shape.STLength(),OMS_FACILITY_ID,PERMIT_ID
0,49347,58133,0,None,9.68121,0,iTEN2015,None,None,None,...,1549323640000,CLAAM,1786479151000,{17D1FAA3-24A5-4D62-8088-B17A3C014755},None,None,3588.883173,280.049324,None,None
1,49424,58210,0,None,11.64050,1959,iTEN2015,None,None,None,...,1549323640000,CLAAM,1578351048000,{1B61EF4C-8DD5-4F6D-AAC1-0A0C5E0B5EB5},None,None,3416.849459,293.436041,None,None
2,49480,58266,0,None,13.43710,0,iTEN2015,None,None,None,...,1549323640000,MCLBR,1549325515000,{5BBCCFE9-FE40-41D4-9148-02624EAAEA9B},None,None,596.723645,101.902240,None,None
3,49587,58374,0,None,15.74650,1976,GEOTERRA2018,None,None,None,...,1549323640000,CLAAM,1552337928000,{D5D3DA8F-0EE5-4B61-AFBF-511A3F73A1DC},None,None,2419.829974,267.729750,None,None
4,49907,58695,0,None,18.47320,1995,GEOTERRA2020,None,None,None,...,1549323640000,CLAAM,1614974080000,{2A419D7F-3DD9-4686-9743-59BD1DF5C374},None,None,3597.114710,263.347027,None,None


In [ ]:
params = {
    "where": "YEAR_BUILT = 0",
    "returnCountOnly": "true",
    "f": "json"
}

response = requests.get(count_url, params=params)
result = response.json()

print("Buildings with YEAR_BUILT = 0:", result["count"])

Buildings with YEAR_BUILT = 0: 8732


In [ ]:
for code, meaning in [("0", "Active"), ("1", "Demoed"), ("2", "Permitted")]:

    params = {
        "where": f"STATUS = '{code}'",
        "returnCountOnly": "true",
        "f": "json"
    }

    response = requests.get(count_url, params=params)
    count = response.json()["count"]

    print(f"{meaning}: {count:,}")

Active: 43,686
Demoed: 0
Permitted: 0


In [ ]:
params = {
    "where": "YEAR_DEMOLISHED IS NOT NULL",
    "returnCountOnly": "true",
    "f": "json"
}

response = requests.get(count_url, params=params)
demolished_year_count = response.json()["count"]

print("Buildings with a demolition year:", demolished_year_count)

Buildings with a demolition year: 0


## Initial Findings

- The layer contains 43,686 building records.
- Geometry type is polygon.
- The current dataset contains no Demoed or Permitted records.
- `YEAR_BUILT = 0` occurs in 8,732 records (approximately 20% of the dataset) and should be treated as a potential missing/unknown value rather than a literal construction year.
- `YEAR_DEMOLISHED` is not populated in the current dataset.
- The `STATUS` field uses a coded domain: 0 = Active, 1 = Demoed, 2 = Permitted.
- The current dataset therefore appears most useful for analyzing the existing building stock rather than historical demolition activity.